# 분할 archive 준비: 합성 검사와 제한된 Drive 순차 읽기

기본 실행은 네트워크 데이터 없이 tiny split tar.gz의 추출·매칭을 검사합니다. 사용자 Colab의 기존 pose 32개 추출 성공과 후속 RGB 0개 실패 이력은 문서에 구분합니다. 이번 소스의 실제 Drive 시험·RGB 대응·학습·GPU는 미검증입니다. M0/M1 완료나 train_ready를 선언하지 않습니다.

소스 ZIP은 64MiB/해제 크기 128MiB 안에서 코드만 풀며 원본 데이터와 다릅니다. 최신 소스를 쓰려면 이전 임시 결과를 먼저 보관하고 커널을 재시작해 새 ZIP을 업로드합니다. 실제 metadata/media는 각각 기본 비활성입니다. [준비 계약](../docs/ARCHIVE_PREPARATION.md), [Colab 안내](../docs/COLAB_SETUP.md)를 따릅니다.


In [ ]:
import sys, os, shutil
print({"python":sys.version.split()[0],"free_disk_bytes":shutil.disk_usage(".").free})


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import io, stat, tempfile, zipfile
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("소스 ZIP 한 개를 선택하세요.")
zip_name, zip_bytes = next(iter(uploaded.items()))
if not zip_name.endswith(".zip") or len(zip_bytes) > 64*1024**2:
    raise ValueError("64MiB 이하의 소스 ZIP 필요")
extract_root = Path(tempfile.mkdtemp(prefix="rewind_source_", dir="/content"))
excluded = {".git", ".venv", "data", "outputs", "checkpoints", ".aws", ".codex", ".agents"}
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
    members = archive.infolist()
    if sum(m.file_size for m in members) > 128*1024**2:
        raise ValueError("소스 ZIP의 해제 크기 상한 초과")
    for member in members:
        p = PurePosixPath(member.filename)
        if p.is_absolute() or ".." in p.parts or any(":" in part for part in p.parts) or stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("잘못된 ZIP member 경로")
        if not any(part in excluded for part in p.parts):
            archive.extract(member, extract_root)
roots = [p.parent.parent for p in extract_root.rglob("rewind_archive/__init__.py")]
if len(roots) != 1:
    raise ValueError("rewind_archive 패키지가 하나인 프로젝트 ZIP 필요")
PROJECT_ROOT = roots[0]
print("source:", PROJECT_ROOT)
stamp = PROJECT_ROOT / "SOURCE_REVISION.txt"
print("revision:", stamp.read_text().strip() if stamp.exists() else "unversioned; code hash로 기록")
os.chdir(PROJECT_ROOT)
sys.path.insert(0,str(PROJECT_ROOT))


In [ ]:
import subprocess
for test_pattern in ("test_archive.py", "test_drive_metadata.py", "test_archive_planning.py"):
    subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-p",test_pattern,"-v"],cwd=PROJECT_ROOT,check=True)


## Tiny 합성 준비

분할 조각은 gzip의 중간에서 나뉘며 하나씩 독립 해제하지 않습니다. 결과와 준비 중 임시 파일은 `/content` 아래 source 프로젝트 outputs에 생성됩니다. Runtime이 삭제되면 소실되므로 필요한 결과·manifest는 별도 영구 보관합니다.


In [ ]:
import datetime, uuid, json
RUN_ROOT = PROJECT_ROOT/"outputs"/("archive_synthetic_"+datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:8])
RUN_ROOT.mkdir(parents=True,exist_ok=False)
subprocess.run([sys.executable,"-m","rewind_archive.fixture","--output",str(RUN_ROOT/"fixture")],cwd=PROJECT_ROOT,check=True)
for kind in ("rgb","pose"):
    subprocess.run([sys.executable,"-m","rewind_archive","extract","--manifest",str(RUN_ROOT/"fixture"/f"{kind}_manifest.json"),
        "--limits",str(PROJECT_ROOT/"configs/archive_limits_smoke.json"),"--include","Fixture/Day/person/avatar/001/Motion__1_/*",
        "--output",str(RUN_ROOT/kind)],cwd=PROJECT_ROOT,check=True)
    subprocess.run([sys.executable,"-m","rewind_archive","verify","--extraction",str(RUN_ROOT/kind),
        "--manifest",str(RUN_ROOT/"fixture"/f"{kind}_manifest.json")],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,"-m","rewind_archive","match","--extraction",str(RUN_ROOT/"rgb"),"--extraction",str(RUN_ROOT/"pose"),
    "--output",str(RUN_ROOT/"matching.json")],cwd=PROJECT_ROOT,check=True)
matching=json.loads((RUN_ROOT/"matching.json").read_text())
print({"matched_triplet_count":matching["matched_triplet_count"],"incomplete_frame_count":matching["incomplete_frame_count"],"train_ready":matching["train_ready"]})
assert matching["matched_triplet_count"]==4 and matching["train_ready"] is False


## 사용자 입력: 한 archive의 000-prefix와 1회 작업

선택 dict/작은 JSON에 index/name/file_id와 공식 total_parts를 입력합니다. Size/version/modifiedTime은 metadata에서 검사합니다. 개인 ID는 저장소에 커밋하지 않습니다. `SELECTED_MEMBER_GLOBS`는 인증 전에 1~32개를 검사합니다. Frame 18개 RGB의 양쪽 camera는 `camera_*`로 표현하면 18개이며 36개 입력은 거부됩니다.

처음 경로를 모르면 `PILOT_PURPOSE="discovery"`와 archive prefix 아래 `*/fisheye_final_image/camera_*/final_*.png` 또는 `*/json/frame_*.json`을 명시합니다. Discovery는 완전 파일 최대2개로 줄이며 다른 예산은 늘리지 않습니다. 관측 파일의 sequence/frame으로만 대응 선택을 제안하며 자동 후속 실행은 없습니다. 0개면 ZIP 보관 후 중단·검토합니다.

입력·metadata 셀을 다시 실행해도 작업 ID와 시작 기록은 유지됩니다. Pilot 1회 시작 후에는 같은 ID로 metadata/media를 다시 호출할 수 없습니다. 아래 별도 **새 작업 선언** 셀만 새 ID를 만들며 비용과 목적 변경 사유를 요구합니다. Gzip 재개 기능이 아니며 새 작업은 byte0부터 읽습니다.


In [ ]:
INPUT_MODE = "dict"  # 또는 "json_upload"
DRIVE_SELECTION = {
    "schema_version":"drive_part_selection_v1",
    "archive_id":"", "dataset_version":"", "kind":"pose", "total_parts":None,
    "parts":[]  # [{"index":0,"name":"USER_ARCHIVE.tar.gz000","file_id":"USER_FILE_ID"}]
}
SELECTED_MEMBER_GLOBS = []  # 인증 전에 1~32개 preflight
PILOT_PURPOSE = "discovery"  # 완전 파일 관측 후 별도 작업은 "targeted"
PILOT_JOB_ID = globals().get("PILOT_JOB_ID", uuid.uuid4().hex)
_PILOT_JOBS = globals().get("_PILOT_JOBS", {})
METADATA_RESULT = None
print({"job_id":PILOT_JOB_ID,"purpose":PILOT_PURPOSE})


## 1. Metadata 확인과 manifest 생성 — 기본 비활성

선택과 glob preflight 후에만 인증합니다. Metadata 기본 파일8/요청8·응답16KiB/누적128KiB·30초/요청5초이며 media와 별도입니다. 검색·자동 retry·token 저장은 하지 않습니다. Metadata 재실행은 이전 result/session을 폐기하고 현재 선택을 검사합니다. 이미 media를 시작한 작업 ID는 metadata 재실행도 차단합니다.

Metadata는 HTTP200·identity·유효 JSON, 길이 헤더 또는 bounded chunked/EOF 종료를 지원합니다. Media는 정확한206·Content-Range·길이를 요구합니다. 최초 인증/refresh 비용은 단계 통계 밖이며 network wire/RSS/OS hard limit이 아닙니다.


In [ ]:
RUN_DRIVE_METADATA = False
METADATA_RESULT = None
METADATA_BUNDLE = None
PILOT_RUN_ROOT = None
if globals().get("DRIVE_SESSION") is not None:
    DRIVE_SESSION.close()
DRIVE_SESSION = None
if RUN_DRIVE_METADATA:
    from rewind_archive.__main__ import read_bounded_json
    from rewind_archive.contracts import PreparationError, validate_includes
    from rewind_archive.drive_metadata import MetadataLimits, validate_selection, build_drive_manifest, save_metadata_bundle
    from rewind_data.audit import read_json
    if INPUT_MODE == "json_upload":
        from google.colab import files
        selection_upload = files.upload()
        if len(selection_upload)!=1 or len(next(iter(selection_upload.values())))>128*1024:
            DRIVE_SELECTION = None
        else:
            try:
                DRIVE_SELECTION = read_json(next(iter(selection_upload.values())))
            except (ValueError, UnicodeError):
                DRIVE_SELECTION = None
    elif INPUT_MODE != "dict":
        DRIVE_SELECTION = None
    validate_includes(SELECTED_MEMBER_GLOBS)
    if PILOT_PURPOSE not in ("discovery","targeted"):
        raise RuntimeError("PILOT_PURPOSE를 확인하세요.")
    if PILOT_JOB_ID in _PILOT_JOBS:
        raise RuntimeError("이미 시작한 작업입니다. 기존 결과를 보관·verify하거나 새 작업 선언을 사용하세요: "+_PILOT_JOBS[PILOT_JOB_ID]["output"])
    metadata_limits = MetadataLimits.from_dict(read_bounded_json(PROJECT_ROOT/"configs/drive_metadata_limits_smoke.json"))
    PILOT_RUN_ROOT = PROJECT_ROOT/"outputs"/("drive_pilot_"+uuid.uuid4().hex)
    valid_input = True
    try:
        validate_selection(DRIVE_SELECTION)
    except (PreparationError, ValueError):
        valid_input = False
    if valid_input:
        from google.colab import auth
        import google.auth
        from google.auth.transport.requests import AuthorizedSession, Request
        from requests.adapters import HTTPAdapter
        import urllib3
        if not callable(getattr(urllib3.response.HTTPResponse,"read1",None)):
            raise RuntimeError("urllib3 raw.read1 API가 필요합니다. 환경을 확인하세요.")
        auth.authenticate_user()
        credentials,_ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
        try:
            credentials.refresh(Request())
        except Exception:
            raise RuntimeError("인증 초기화 실패. 인증 환경을 확인하세요.") from None
        DRIVE_SESSION = AuthorizedSession(credentials,max_refresh_attempts=0)
        DRIVE_SESSION.mount("https://",HTTPAdapter(max_retries=0))
    METADATA_RESULT = build_drive_manifest(DRIVE_SELECTION,DRIVE_SESSION,metadata_limits)
    METADATA_BUNDLE = save_metadata_bundle(METADATA_RESULT,PILOT_RUN_ROOT/"metadata")
    r=METADATA_RESULT["report"]
    print({"manifest_ready":r["manifest_ready"],"stop_reason":r["stop_reason"],"stats":r["stats"],
        "media_requested_by_builder":r["media_requested_by_builder"],"train_ready":False,"output":str(METADATA_BUNDLE)})
else:
    print("실제 Drive 인증/metadata 조회를 실행하지 않았습니다.")


## 2. Bounded pilot — 기본 비활성, 작업당 1회

본문 합계8MiB·60초·logical disk16MiB·총 요청128을 유지합니다. HTTP media Range256KiB, 내부 read1/해제 IO64KiB입니다. Drive는 media마다 전후 metadata를 검사하고 시작 전에 3요청 및 metadata32KiB를 예약합니다. Metadata 실제 body가 더 작으면 잔여 bytes를 다음 Range에 사용할 수 있지만 마지막 약32KiB는 보수적으로 남을 수 있습니다. 후검사가 미완료면 후보 전체를 unusable로 처리합니다.

`ranges`는 기존 총 source 동작 수이며 `media_requests`, `metadata_requests`, `http_requests_total`, `local_reads`를 분리합니다. `sample_acquired=false`와 0개는 verify가 passed여도 샘플 확보 성공이 아닙니다. 상한·오류·0개면 ZIP을 보관하고 멈춥니다. 자동 재시도·상한 확대·학습·원본 전체 다운로드는 없습니다.


In [ ]:
RUN_DRIVE_PILOT = False
PILOT_REPORT = globals().get("PILOT_REPORT")
if RUN_DRIVE_PILOT:
    from dataclasses import replace
    from rewind_archive.contracts import Limits, digest, validate_includes
    from rewind_archive.drive_metadata import validate_selection
    from rewind_archive.pilot import run_metadata_pilot
    from rewind_archive.planning import claim_pilot_job
    from rewind_archive.__main__ import read_bounded_json
    if METADATA_RESULT is None or DRIVE_SESSION is None or PILOT_RUN_ROOT is None:
        raise RuntimeError("유효한 metadata 단계를 먼저 실행하세요.")
    validate_selection(DRIVE_SELECTION)
    validate_includes(SELECTED_MEMBER_GLOBS)
    if METADATA_RESULT["report"]["selection_sha256"] != digest(DRIVE_SELECTION):
        raise RuntimeError("선택 입력이 바뀌었습니다. Metadata를 다시 확인하세요.")
    extraction_limits = Limits.from_dict(read_bounded_json(PROJECT_ROOT/"configs/archive_limits_smoke.json"))
    if PILOT_PURPOSE == "discovery":
        extraction_limits = replace(extraction_limits,max_files=min(2,extraction_limits.max_files))
    claim_pilot_job(_PILOT_JOBS,PILOT_JOB_ID,DRIVE_SELECTION,SELECTED_MEMBER_GLOBS,
        extraction_limits,PILOT_PURPOSE,PILOT_RUN_ROOT)
    LAST_PILOT_RUN_ROOT = PILOT_RUN_ROOT
    PILOT_REPORT = None
    (PILOT_RUN_ROOT/"pilot_job.json").write_text(json.dumps({"job_id":PILOT_JOB_ID,**_PILOT_JOBS[PILOT_JOB_ID]},indent=2),encoding="utf-8")
    print("extraction_limits:",json.dumps(extraction_limits.__dict__,indent=2))
    PILOT_REPORT = run_metadata_pilot(METADATA_RESULT,PILOT_RUN_ROOT/"pilot",SELECTED_MEMBER_GLOBS,extraction_limits,DRIVE_SESSION)
    print({k:PILOT_REPORT[k] for k in ("complete_files","sample_status","sample_acquired","verification","stats","metadata_stage_stats",
        "stop_reason","failure_class","source_validation_pending","gzip_eof_crc_valid","whole_archive_integrity_verified","train_ready","pending_verifications")})
    print("local bundle:",PILOT_RUN_ROOT)
else:
    print("실제 Drive media/pilot을 실행하지 않았습니다.")


## 3. 결과 ZIP 보관 — 원본을 읽지 않음

실패·0개도 metadata/extraction/pilot 보고서를 보관합니다. 아래는 이미 생성한 작은 작업 디렉터리만 ZIP으로 만듭니다. 원본 Drive 압축·소스 ZIP·전체 outputs는 넣지 않습니다. 기본값은 저장하지 않으며 다운로드도 별도 선택입니다. 런타임 종료 전에 ZIP을 내려받아 보관합니다. ZIP은 개인 file ID와 sample을 포함할 수 있으므로 공개 게시하지 않습니다.


In [ ]:
SAVE_RESULT_ZIP = False
DOWNLOAD_RESULT_ZIP = False
if SAVE_RESULT_ZIP:
    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    result_root = globals().get("PILOT_RUN_ROOT") or globals().get("LAST_PILOT_RUN_ROOT")
    if result_root is None or not Path(result_root).is_dir():
        raise RuntimeError("저장할 작업 디렉터리가 없습니다.")
    result_root = Path(result_root).resolve()
    if result_root.parent != (PROJECT_ROOT/"outputs").resolve() or not result_root.name.startswith("drive_pilot_"):
        raise RuntimeError("작업 결과 경로를 확인하세요.")
    result_files = [p for p in result_root.rglob("*") if p.is_file()]
    if any(p.is_symlink() or not p.resolve().is_relative_to(result_root) for p in result_root.rglob("*")):
        raise RuntimeError("결과 경로의 link를 거부합니다.")
    if sum(p.stat().st_size for p in result_files)>17*1024**2:
        raise RuntimeError("작은 결과 bundle 상한을 초과했습니다.")
    RESULT_ZIP_PATH = result_root.with_suffix(".zip")
    if RESULT_ZIP_PATH.is_symlink():
        raise RuntimeError("결과 ZIP link를 거부합니다.")
    if RESULT_ZIP_PATH.exists():
        print("기존 ZIP을 유지합니다. 이후 생성된 파일은 자동 추가하지 않습니다.")
    else:
        with ZipFile(RESULT_ZIP_PATH,"x",ZIP_DEFLATED) as bundle:
            for p in result_files:
                bundle.write(p,str(Path(result_root.name)/p.relative_to(result_root)))
    print("result ZIP:",RESULT_ZIP_PATH)
else:
    print("결과 ZIP을 새로 저장하지 않았습니다.")
if DOWNLOAD_RESULT_ZIP:
    from pathlib import Path
    if not globals().get("RESULT_ZIP_PATH") or not Path(RESULT_ZIP_PATH).is_file() or Path(RESULT_ZIP_PATH).is_symlink():
        raise RuntimeError("먼저 결과 ZIP을 저장하세요.")
    from google.colab import files
    files.download(str(RESULT_ZIP_PATH))


## 4. 완전 파일 관측 → 대응 선택 제안 → 재사용 또는 중단

다음 셀은 원본에 접근하지 않고 기존 추출본의 hash를 확인한 뒤 실제 관측 sequence/frame으로 RGB/pose glob을 제안합니다. 제안은 화면에만 표시하며 앞서 저장한 ZIP에 자동 추가하지 않습니다. 선택 경로 뒤쪽이 현재 예산 안에 도달할지는 unknown입니다. 0개면 제안 없이 중단합니다. 완전 파일과 잘린 active member는 구분합니다. 기존 파일의 byte hash 확인이 목적이며 PNG/JSON 내용·시간·split·원본 motion 검증은 별도입니다.

다른 archive의 대응 파일이 필요하면 결과를 검토해 별도 신규 작업으로만 실행합니다. 현재 추출본을 verify해 재사용하고, 중간 gzip 상태 재개나 같은 RGB 작업의 반복을 기대하지 않습니다. 여러 추출본은 `match`로 대조하며 matching_valid와 triplet 수를 함께 확인합니다. Train_ready는 계속 false입니다.


In [ ]:
MAKE_COUNTERPART_PLAN = False
if MAKE_COUNTERPART_PLAN:
    from rewind_archive.planning import counterpart_plan
    if not globals().get("LAST_PILOT_RUN_ROOT"):
        raise RuntimeError("기존 추출 결과가 필요합니다.")
    COUNTERPART_PLAN = counterpart_plan([LAST_PILOT_RUN_ROOT/"pilot"],max_frames=8)
    print(json.dumps(COUNTERPART_PLAN,ensure_ascii=False,indent=2))


## 5. 새 작업 선언 — 명시적 비용·목적 검토 후에만

기존 ZIP 보관·파일 재사용·실패 원인을 먼저 검토합니다. 아래 두 변수를 직접 바꾸면 새 job ID를 만들고 오래된 metadata/session을 폐기합니다. 다음 선택/glob·목적을 수정해 metadata부터 실행합니다. 같은 입력도 byte0부터 다시 읽으며, 이전 작업 기록은 삭제하지 않습니다. 이 셀은 retry나 sampler resume가 아닙니다. 기본 비활성이며 실행 후 토글은 자동으로 False로 돌아갑니다.


In [ ]:
START_NEW_PILOT_JOB = False
NEW_JOB_REASON = ""  # 왜 기존 결과 재사용/중단으로 충분하지 않은지 명시
if START_NEW_PILOT_JOB:
    if not isinstance(NEW_JOB_REASON,str) or not NEW_JOB_REASON.strip():
        raise RuntimeError("새 작업의 비용과 목적 변경 사유가 필요합니다.")
    if globals().get("DRIVE_SESSION") is not None:
        DRIVE_SESSION.close()
    DRIVE_SESSION = METADATA_RESULT = METADATA_BUNDLE = PILOT_RUN_ROOT = None
    PILOT_JOB_ID = uuid.uuid4().hex
    START_NEW_PILOT_JOB = False
    print({"new_job_id":PILOT_JOB_ID,"reason":NEW_JOB_REASON,"reads_from_byte_zero":True})
